[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_2_Whole_Columns.ipynb)

# 06.2: Working on a Whole Column

In notebook 06.1 you created a series and looked up values one at a time. Most questions about data involve every value, though, such as "which majors pay more than $50,000?" With a list you would write a loop for that. This notebook shows how a series lets you do math, comparisons, filtering and sorting on all of its values at once, and how to change values safely.

## The data

This uses the same majors data as 06.1, with one new series. `share_women` holds the fraction of graduates in each major who are women, from 0 to 1. Both series have the major names as their index.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
share_women = pd.Series(majors["ShareWomen"].to_list(), index=major_list)
salaries.head(3)

In [ ]:
share_women.head(3)

The share for PETROLEUM ENGINEERING is about 0.12, so about 12 percent of those graduates are women.

## Math without a loop

Suppose you want the salaries in thousands of dollars. With a list, you would write a loop and build a new list.

In [ ]:
in_thousands_list = []
for salary in salaries.to_list():
    in_thousands_list.append(salary / 1000)
in_thousands_list[:3]

With a series, you divide the whole series by 1000. pandas applies the operation to every value and keeps every label. This is called a **vectorized** operation.

In [ ]:
salaries / 1000

PETROLEUM ENGINEERING became 110.0 and LIBRARY SCIENCE became 22.0. The same idea works for the other arithmetic operators, so `salaries + 1000` adds 1000 to every value.

## Results are new series

An operation like this does not change the original series. It produces a new series, which is displayed and then lost unless you assign it to a variable. The next cell shows that `salaries` still holds the original numbers.

In [ ]:
in_thousands = salaries / 1000
print(in_thousands.iloc[0])
print(salaries.iloc[0])

`in_thousands` starts with 110.0, and `salaries` still starts with 110000. The same is true of `.sort_values()` from 06.1: it gives you a new sorted series and leaves the original in its old order. To keep the result, assign it to a name.

## Comparisons and boolean series

A comparison also works on every value, and it produces a series of `True` and `False` values with the same labels.

In [ ]:
salaries > 70000

A series of booleans like this one is called a **boolean mask**. On its own it is hard to read, since only a few of the 173 values are `True`. There are two useful things to do with it. The first is to count: `True` counts as 1 and `False` counts as 0, so `.sum()` gives the number of `True` values.

In [ ]:
(salaries > 70000).sum()

Three majors have a median above 70000. The second is to filter: when you put a mask inside the square brackets, pandas keeps only the items where the mask is `True`.

In [ ]:
salaries[salaries > 70000]

Those three are PETROLEUM ENGINEERING (110000), MINING AND MINERAL ENGINEERING (75000) and METALLURGICAL ENGINEERING (73000). Read `salaries[salaries > 70000]` from the inside out: first build the mask, then use it to select from the series. The result is a new series, so `salaries` itself still has all 173 items.

Watch the difference between `>` and `>=`. In this data 17 majors have a median of exactly 40000, so `salaries > 40000` and `salaries >= 40000` give different answers.

In [ ]:
print((salaries > 40000).sum())
print((salaries >= 40000).sum())

## Combining conditions

Because the two series have the same index, a mask built from one can be used to filter the other, and masks can be combined. Use `&` for and, `|` for or, and `~` for not. Each condition needs its own parentheses, and the Python words `and`, `or` and `not` will not work here, because they expect a single `True` or `False` and not a whole series of them.

Which majors have a median above $50,000 and are mostly women?

In [ ]:
salaries[(salaries > 50000) & (share_women > 0.5)]

Only ASTRONOMY AND ASTROPHYSICS, at 62000, meets both conditions, out of the 25 majors above $50,000 and the 96 majors that are mostly women. Now an example with `|`: which majors have a median that is either above $70,000 or below $25,000?

In [ ]:
salaries[(salaries > 70000) | (salaries < 25000)]

This returns the three majors from before and also COUNSELING PSYCHOLOGY (23400) and LIBRARY SCIENCE (22000). `~` flips a mask, turning every `True` into `False` and the reverse.

In [ ]:
(~(share_women > 0.5)).sum()

That gives 77 majors, which together with the 96 above makes all 173. One of the 77 is FOOD SCIENCE, whose share is missing from the data. A comparison with a missing value gives `False`, so `~` turns it into `True`, and FOOD SCIENCE gets counted even though we do not know its share. Notebook 06.3 returns to missing values.

To check whether values match any item in a list, use `.isin()`, which is shorter than chaining many `|` conditions. This works on any series, and it also makes a mask.

In [ ]:
salaries[salaries.isin([110000, 75000])]

## Sorting

`.sort_values()` sorts by value, and `.sort_index()` sorts by label. Both sort from low to high unless you pass `ascending=False`. Combining a descending sort with `.head()` gives you a top-N list.

In [ ]:
salaries.sort_values(ascending=False).head(3)

In [ ]:
salaries.sort_index().head(3)

The first result lists the three highest medians, and the second lists the first three majors alphabetically: ACCOUNTING, ACTUARIAL SCIENCE and ADVERTISING AND PUBLIC RELATIONS. Chaining like this works because `.sort_values()` returns a series, and you can call `.head()` on that series.

## Copies and aliases

Assigning a series to another name does not copy it, the same as with lists. Both names refer to one object, which is called an **alias**. To get an independent series, use `.copy()`.

In [ ]:
alias = salaries
independent = salaries.copy()
print(alias is salaries)
print(independent is salaries)

`alias` is the same object as `salaries`, and `independent` is a different one. This matters as soon as you change a value.

## Changing values

Unlike the operations above, assigning with `.loc` changes the series directly. Here we work on a copy so that `salaries` stays as it was.

In [ ]:
updated = salaries.copy()
updated.loc["NURSING"] = 50000
print(updated.loc["NURSING"])
print(salaries.loc["NURSING"])

The copy now has 50000 and `salaries` still has 48000. If we had written `updated = salaries`, without `.copy()`, then the change would have shown up under both names. You can also change many values at once by putting a mask inside `.loc`. The next cell raises every salary below 25000 to 25000 in a copy.

In [ ]:
floored = salaries.copy()
floored.loc[floored < 25000] = 25000
floored.tail(3)

COUNSELING PSYCHOLOGY and LIBRARY SCIENCE are now both 25000, and CLINICAL PSYCHOLOGY stays at 25000 because it was not below the cutoff.

## What's next

You can now do math on a whole series, filter it with masks, sort it and change it safely. So far you have mostly picked out individual items, but often you want a summary of the whole column, such as what a typical salary is and how spread out the salaries are. Notebook 06.3 introduces the descriptive statistics that answer those questions.